# Implementasi Webinar Pertemuan 4
## Introduction to Machine Learning — Prediksi Performa Akademik Mahasiswa

**Platform:** Python Google Colab  
**Tingkat kompleksitas:** Menengah  
**Studi kasus:** Prediksi nilai akhir mahasiswa berdasarkan jam belajar, kehadiran, nilai tugas, dan nilai kuis.

> **Catatan:** Dataset pada notebook ini merupakan data simulasi untuk pembelajaran, bukan data mahasiswa nyata. Hasil model tidak boleh digunakan untuk mengambil keputusan akademik nyata.

## 1. Latar Belakang dan Tujuan

*Machine Learning* memungkinkan komputer mempelajari pola dari data untuk menghasilkan prediksi. Pada studi kasus ini, kita akan memprediksi nilai akhir mahasiswa menggunakan pendekatan **supervised learning — regression**, karena target yang diprediksi berupa angka.

Tujuan implementasi:
1. Membuat dan memahami dataset simulasi.
2. Memeriksa kualitas data dan melakukan prapemrosesan.
3. Membagi data menjadi data latih dan data uji.
4. Melatih dan membandingkan model *Linear Regression* dan *Random Forest Regressor*.
5. Mengevaluasi model dengan MAE, RMSE, dan R².
6. Memeriksa indikasi *overfitting*, memahami fitur yang berpengaruh, dan membuat prediksi untuk data baru.
7. Membahas keterbatasan serta etika penggunaan model.

## 2. Memahami Konsep Utama

- **AI:** bidang luas untuk membuat sistem yang melakukan tugas yang membutuhkan kecerdasan.
- **Machine Learning:** bagian dari AI yang mempelajari pola dari data.
- **Deep Learning:** bagian dari *Machine Learning* yang menggunakan jaringan saraf berlapis.
- **Supervised learning:** belajar dari data yang memiliki target/label.
- **Regression:** memprediksi nilai numerik, misalnya nilai akhir.
- **Training data:** data untuk melatih model.
- **Testing data:** data terpisah untuk mengukur kinerja pada contoh yang belum dilihat model.

Alur notebook: simulasi data → pemeriksaan kualitas → eksplorasi → pemisahan data → pelatihan → evaluasi → interpretasi → prediksi baru.

## 3. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Library berhasil diimpor.")

## 4. Membuat Dataset Simulasi

Dataset dibuat untuk 500 mahasiswa dengan fitur:
- `jam_belajar_per_minggu`
- `kehadiran_persen`
- `nilai_tugas`
- `nilai_kuis`
- `nilai_akhir` sebagai target prediksi

Nilai akhir dibentuk dari kombinasi beberapa fitur dan sedikit variasi acak agar data simulasi lebih realistis. Rumus ini hanya asumsi untuk latihan, bukan aturan penilaian akademik yang berlaku umum.

In [ ]:
n = 500

jam_belajar = np.clip(np.random.normal(loc=8, scale=3, size=n), 0.5, 20)
kehadiran = np.clip(np.random.normal(loc=85, scale=9, size=n), 50, 100)
nilai_tugas = np.clip(np.random.normal(loc=75, scale=12, size=n), 30, 100)
nilai_kuis = np.clip(np.random.normal(loc=72, scale=14, size=n), 20, 100)
noise = np.random.normal(loc=0, scale=5, size=n)

nilai_akhir = (
    18
    + 1.25 * jam_belajar
    + 0.18 * kehadiran
    + 0.28 * nilai_tugas
    + 0.22 * nilai_kuis
    + noise
)
nilai_akhir = np.clip(nilai_akhir, 0, 100)

df = pd.DataFrame({
    "jam_belajar_per_minggu": np.round(jam_belajar, 1),
    "kehadiran_persen": np.round(kehadiran, 1),
    "nilai_tugas": np.round(nilai_tugas, 1),
    "nilai_kuis": np.round(nilai_kuis, 1),
    "nilai_akhir": np.round(nilai_akhir, 1)
})

df.head()

### 4.1 Melihat ukuran dan ringkasan dataset

In [ ]:
print(f"Jumlah baris: {df.shape[0]}")
print(f"Jumlah kolom: {df.shape[1]}")
display(df.describe().round(2))

## 5. Pemeriksaan Kualitas Data

Tahap ini mencerminkan prinsip webinar bahwa kualitas data memengaruhi kualitas model. Kita memeriksa tipe data, nilai kosong, duplikasi, dan rentang nilai. Untuk mendemonstrasikan penanganan data kosong, beberapa nilai fitur sengaja dibuat kosong pada salinan dataset.

In [ ]:
print("Tipe data:")
display(df.dtypes.to_frame("tipe_data"))

print("Jumlah nilai kosong per kolom:")
display(df.isna().sum().to_frame("jumlah_kosong"))

print("Jumlah baris duplikat:", df.duplicated().sum())

# Salinan khusus untuk demonstrasi pembersihan data
data = df.copy()
missing_rows = np.random.choice(data.index, size=15, replace=False)
data.loc[missing_rows[:8], "jam_belajar_per_minggu"] = np.nan
data.loc[missing_rows[8:], "nilai_tugas"] = np.nan

print("\nSetelah simulasi nilai kosong:")
display(data.isna().sum().to_frame("jumlah_kosong"))

### 5.1 Visualisasi distribusi target dan hubungan fitur

Grafik berikut membantu memahami distribusi nilai akhir serta hubungan awal antara jam belajar dan nilai akhir. Hubungan yang terlihat pada data simulasi tidak membuktikan hubungan sebab-akibat.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df["nilai_akhir"], bins=20, edgecolor="black")
axes[0].set_title("Distribusi Nilai Akhir")
axes[0].set_xlabel("Nilai akhir")
axes[0].set_ylabel("Jumlah mahasiswa")

axes[1].scatter(df["jam_belajar_per_minggu"], df["nilai_akhir"], alpha=0.6)
axes[1].set_title("Jam Belajar dan Nilai Akhir")
axes[1].set_xlabel("Jam belajar per minggu")
axes[1].set_ylabel("Nilai akhir")

plt.tight_layout()
plt.show()

## 6. Menentukan Fitur dan Target serta Membagi Data

Target (`y`) adalah `nilai_akhir`, sedangkan fitur (`X`) adalah empat variabel prediktor. Data dibagi menjadi 80% data latih dan 20% data uji. Data uji tidak digunakan untuk melatih model sehingga evaluasi lebih mencerminkan kemampuan model pada data yang belum dilihat.

In [ ]:
features = [
    "jam_belajar_per_minggu",
    "kehadiran_persen",
    "nilai_tugas",
    "nilai_kuis"
]
target = "nilai_akhir"

X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Ukuran X_train:", X_train.shape)
print("Ukuran X_test :", X_test.shape)
print("Proporsi data latih:", round(len(X_train) / len(data) * 100, 1), "%")
print("Proporsi data uji  :", round(len(X_test) / len(data) * 100, 1), "%")

## 7. Melatih Dua Model

Kita membandingkan:
1. **Linear Regression** — model dasar yang mempelajari hubungan linear antara fitur dan target. Fitur numerik distandardisasi; nilai kosong diisi dengan median.
2. **Random Forest Regressor** — gabungan banyak pohon keputusan yang dapat menangkap pola nonlinier. Model ini tidak membutuhkan standardisasi, tetapi tetap menggunakan imputasi median.

Semua langkah prapemrosesan diletakkan dalam *pipeline* agar proses pelatihan konsisten dan mengurangi risiko kebocoran data (*data leakage*).

In [ ]:
linear_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

forest_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=3,
        random_state=RANDOM_STATE
    ))
])

linear_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

print("Kedua model selesai dilatih.")

## 8. Evaluasi Model

Metrik yang digunakan:
- **MAE (*Mean Absolute Error*):** rata-rata selisih absolut antara prediksi dan nilai sebenarnya. Semakin kecil semakin baik.
- **RMSE (*Root Mean Squared Error*):** memberi penalti lebih besar pada kesalahan yang besar. Semakin kecil semakin baik.
- **R²:** proporsi variasi target yang dapat dijelaskan model pada data evaluasi. Nilai mendekati 1 umumnya lebih baik; nilai negatif berarti model lebih buruk daripada prediksi sederhana menggunakan rata-rata target pada data uji.

In [ ]:
def evaluate_regression(model, X_part, y_part, model_name, split_name):
    pred = model.predict(X_part)
    return {
        "Model": model_name,
        "Data": split_name,
        "MAE": mean_absolute_error(y_part, pred),
        "RMSE": np.sqrt(mean_squared_error(y_part, pred)),
        "R2": r2_score(y_part, pred)
    }

results = []
for name, model in [
    ("Linear Regression", linear_model),
    ("Random Forest", forest_model)
]:
    results.append(evaluate_regression(model, X_test, y_test, name, "Test"))

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df.round(3))

### 8.1 Membandingkan nilai aktual dan prediksi

Grafik menunjukkan seberapa dekat prediksi model terhadap nilai aktual. Titik yang semakin dekat dengan garis diagonal ideal menunjukkan prediksi yang semakin mendekati nilai sebenarnya.

In [ ]:
pred_linear = linear_model.predict(X_test)
pred_forest = forest_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pred, title in [
    (axes[0], pred_linear, "Linear Regression"),
    (axes[1], pred_forest, "Random Forest")
]:
    ax.scatter(y_test, pred, alpha=0.7)
    ax.plot([0, 100], [0, 100], linestyle="--")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xlabel("Nilai aktual")
    ax.set_ylabel("Nilai prediksi")
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 9. Memeriksa Indikasi Overfitting

Kita membandingkan kinerja pada data latih dan data uji. Jika hasil data latih jauh lebih baik daripada data uji, model mungkin terlalu menyesuaikan diri dengan data latih. Perbedaan metrik saja bukan diagnosis mutlak; perlu dilihat bersama karakteristik data dan metode validasi.

In [ ]:
overfit_rows = []
for name, model in [
    ("Linear Regression", linear_model),
    ("Random Forest", forest_model)
]:
    for split_name, X_part, y_part in [
        ("Train", X_train, y_train),
        ("Test", X_test, y_test)
    ]:
        row = evaluate_regression(model, X_part, y_part, name, split_name)
        overfit_rows.append(row)

overfit_df = pd.DataFrame(overfit_rows)
display(overfit_df.round(3))

print("Interpretasi awal:")
for name in ["Linear Regression", "Random Forest"]:
    sub = overfit_df[overfit_df["Model"] == name].set_index("Data")
    gap = sub.loc["Test", "RMSE"] - sub.loc["Train", "RMSE"]
    print(f"- {name}: selisih RMSE test - train = {gap:.3f}")

## 10. Interpretasi Fitur

Untuk *Random Forest*, kita dapat melihat *feature importance* sebagai gambaran seberapa banyak fitur berkontribusi terhadap pembagian pada pohon-pohon model. Nilai ini bukan bukti sebab-akibat dan dapat dipengaruhi korelasi antarfitur.

In [ ]:
rf = forest_model.named_steps["model"]
importance = pd.DataFrame({
    "Fitur": features,
    "Feature Importance": rf.feature_importances_
}).sort_values("Feature Importance", ascending=False)

display(importance.round(4))

plt.figure(figsize=(8, 4))
plt.barh(importance["Fitur"][::-1], importance["Feature Importance"][::-1])
plt.title("Feature Importance — Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 11. Prediksi untuk Data Mahasiswa Baru

Contoh ini menunjukkan cara memasukkan data baru ke model. Hasil prediksi merupakan perkiraan berdasarkan pola dataset simulasi, bukan nilai resmi atau penilaian mahasiswa yang sebenarnya.

In [ ]:
mahasiswa_baru = pd.DataFrame([{
    "jam_belajar_per_minggu": 10,
    "kehadiran_persen": 90,
    "nilai_tugas": 82,
    "nilai_kuis": 78
}])

prediksi_linear = linear_model.predict(mahasiswa_baru)[0]
prediksi_forest = forest_model.predict(mahasiswa_baru)[0]

print(f"Prediksi Linear Regression: {prediksi_linear:.2f}")
print(f"Prediksi Random Forest    : {prediksi_forest:.2f}")
print("\nGunakan hasil ini sebagai demonstrasi teknis, bukan keputusan akademik.")

## 12. Interpretasi Hasil Otomatis

Sel berikut merangkum model dengan RMSE terkecil pada data uji. Model terbaik untuk dataset simulasi ini belum tentu terbaik untuk data nyata atau dataset lain.

In [ ]:
best_row = results_df.iloc[0]
print(f"Model dengan RMSE terkecil pada data uji: {best_row['Model']}")
print(f"MAE  : {best_row['MAE']:.3f}")
print(f"RMSE : {best_row['RMSE']:.3f}")
print(f"R²   : {best_row['R2']:.3f}")

print("\nCara membaca hasil:")
print("- MAE dan RMSE dinyatakan dalam satuan nilai akhir.")
print("- R² menggambarkan kecocokan model terhadap variasi target pada data uji.")
print("- Bandingkan metrik bersama grafik aktual vs prediksi dan selisih train-test.")

## 13. Tantangan, Keterbatasan, dan Etika

Beberapa hal yang perlu diperhatikan:
1. **Data simulasi:** pola target sengaja dibentuk dari rumus tertentu. Kinerja pada data nyata dapat berbeda.
2. **Bias dan faktor yang tidak terukur:** kondisi ekonomi, akses perangkat, kualitas pengajaran, kesehatan, dan banyak faktor lain tidak tercakup.
3. **Korelasi bukan kausalitas:** fitur yang penting bagi model tidak otomatis menjadi penyebab nilai akhir.
4. **Privasi:** data pendidikan nyata perlu dikelola dengan izin dan perlindungan yang sesuai.
5. **Akuntabilitas:** model tidak boleh menjadi satu-satunya dasar keputusan akademik penting. Manusia harus meninjau konteks dan kemungkinan kesalahan.
6. **Pemantauan:** jika digunakan secara nyata, model perlu diuji ulang ketika pola data berubah.

Untuk memperkuat evaluasi pada proyek lanjutan, gunakan *cross-validation*, periksa kualitas dan representativitas data, serta dokumentasikan asumsi model.

## 14. Kesimpulan

Pada notebook ini, alur dasar *Machine Learning* diterapkan melalui studi kasus prediksi nilai akhir mahasiswa. Tahap yang dilakukan meliputi pembuatan dataset simulasi, pemeriksaan kualitas data, pemisahan data latih dan data uji, pelatihan dua algoritma regresi, evaluasi menggunakan MAE/RMSE/R², pemeriksaan indikasi *overfitting*, interpretasi fitur, dan prediksi data baru.

Implementasi ini sesuai dengan pokok Webinar ke-4: pemilihan algoritma perlu disesuaikan dengan tujuan, kualitas data perlu diperiksa, evaluasi harus dilakukan pada data yang tidak digunakan untuk pelatihan, dan penggunaan model harus memperhatikan keterbatasan serta etika.

**Pengembangan berikutnya:** coba dataset nyata yang diperoleh secara sah, gunakan *cross-validation*, bandingkan model tambahan, dan dokumentasikan sumber data serta batasan analisis.